# Stance Training on Google Colab
Train the small two-stage stance pipeline on a Colab T4, then download the final checkpoint.

## What this notebook does
- clones your repo
- installs only the training dependencies
- builds `stage1_public_small`
- trains `stage1_public_small`
- trains `stage2_hardcases_small`
- zips the final checkpoint for download

In [1]:
# Optional: mount Google Drive to persist outputs
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/fact_checking_system_colab'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import shutil

REPO_URL = "https://github.com/injetiharsha/fact_checking_system.git"
BRANCH = "feat/reduce-heuristics-phased"
REPO_DIR = "/content/fact_checking_system"

os.chdir("/content")

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


Cloning into '/content/fact_checking_system'...
remote: Enumerating objects: 709, done.
remote: Counting objects: 100% (709/709), done.
remote: Compressing objects: 100% (381/381), done.
remote: Total 709 (delta 347), reused 663 (delta 301), pack-reused 0 (from 0)
Receiving objects: 100% (709/709), 749.58 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (347/347), done.
cwd: /content/fact_checking_system


In [3]:
# Training-focused dependencies only
!pip install -q --upgrade pip
!pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.38.2 datasets==2.17.1 accelerate==0.27.2 scikit-learn==1.4.2 numpy==1.26.4 scipy==1.12.0 PyYAML==6.0.2 sentencepiece==0.2.0 tqdm==4.66.2 requests==2.31.0 urllib3==1.26.18


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.6 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.31.0 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.12.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have nump

In [4]:
!pip install -q --upgrade pip
!pip uninstall -y peft bitsandbytes

!pip install -q \
  transformers==4.38.2 \
  datasets==2.17.1 \
  accelerate==0.27.2 \
  scikit-learn==1.4.2 \
  sentencepiece==0.2.0 \
  PyYAML==6.0.2 \
  tqdm==4.66.2


Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [5]:
import os, torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
os.environ['PYTHONWARNINGS'] = 'ignore'


CUDA available: True
CUDA device: Tesla T4


In [6]:
# Build the reduced public stage-1 dataset
!python training/common/build_stance_stage1_public.py

Generating train split: 370653 examples [00:01, 270330.19 examples/s]
Generating validation split: 63054 examples [00:00, 264692.90 examples/s]
Generating test split: 55197 examples [00:00, 107643.06 examples/s]
Generating train_r1 split: 100% 16946/16946 [00:00<00:00, 448339.94 examples/s]
Generating dev_r1 split: 100% 1000/1000 [00:00<00:00, 259243.71 examples/s]
Generating test_r1 split: 100% 1000/1000 [00:00<00:00, 275995.53 examples/s]
Generating train_r2 split: 100% 45460/45460 [00:00<00:00, 1001139.68 examples/s]
Generating dev_r2 split: 100% 1000/1000 [00:00<00:00, 268504.19 examples/s]
Generating test_r2 split: 100% 1000/1000 [00:00<00:00, 306915.26 examples/s]
Generating train_r3 split: 100% 100459/100459 [00:00<00:00, 1039473.02 examples/s]
Generating dev_r3 split: 100% 1200/1200 [00:00<00:00, 259200.99 examples/s]
Generating test_r3 split: 100% 1200/1200 [00:00<00:00, 323551.35 examples/s]
Generating train split: 100% 392702/392702 [00:00<00:00, 726795.98 examples/s]
Genera

In [ ]:
# Stage 1 small
!python -u training/stance/train.py --config training/configs/stance_stage1_public_small.yaml

2026-03-15 12:48:01.251135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773578881.272235    7333 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773578881.279127    7333 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773578881.296593    7333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773578881.296618    7333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773578881.296621    7333 computation_placer.cc:177] computation placer alr

In [ ]:
# Stage 2 hardcases small
!python -u training/stance/train.py --config training/configs/stance_stage2_hardcases_small.yaml

In [ ]:
# Package final checkpoint and metrics
FINAL_DIR = 'checkpoints/stance/stage2_hardcases_small'
METRICS_DIR = 'training_artifacts/stance/stage2_hardcases_small'
ARCHIVE = '/content/stance_stage2_hardcases_small.zip'
!zip -r {ARCHIVE} {FINAL_DIR} {METRICS_DIR}
print('Created:', ARCHIVE)

In [ ]:
# Optional: copy archive to Drive
if USE_DRIVE:
    import os, shutil
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy('/content/stance_stage2_hardcases_small.zip', os.path.join(DRIVE_DIR, 'stance_stage2_hardcases_small.zip'))
    print('Copied archive to', DRIVE_DIR)

In [ ]:
# Download to your machine
from google.colab import files
files.download('/content/stance_stage2_hardcases_small.zip')